# Qwen3-4B on Spider dev: RAG and ReAct ablation

This notebook reruns the whole experiment end to end: the baseline run plus the five extension arms, then the comparison tables and sampled traces. Every step calls the scripts in `agent_ext/`, so results match runs made from the command line.

**Arms.** Each arm is one full run of Qwen3-4B (bf16, greedy decoding, thinking off, batch size 64) over the 1034 Spider dev questions. A ✓ means that component is on.

| arm | few-shot (5 nearest train examples) | ReAct loop | act tools |
|---|---|---|---|
| `base` | – | – | – (the original A1 prompt) |
| `rag` | ✓ | – | – |
| `react` | – | ✓ | `execute` |
| `react_parse` | – | ✓ | `execute` + `spider_parse` |
| `rag_react` | ✓ | ✓ | `execute` |
| `rag_react_parse` | ✓ | ✓ | `execute` + `spider_parse` |

Compare two arms that differ in one column to see the effect of that component. For example, `rag` vs `base` is the effect of RAG, and `rag_react` vs `rag` is what ReAct adds on top of RAG.

- `execute` runs the query on a schema-only SQLite database, which has no rows. It reports the result columns or the SQLite error.
- `spider_parse` reports whether Spider's grader can parse the query.
- Few-shot examples come from Spider **train**, which shares no databases with dev.

## 0. One-time setup (login node, needs internet)

Run this from the repo root on a machine with internet access. It creates `.venv`, the schema-only Spider databases, NLTK data, and the model weights (Qwen3-4B and the all-MiniLM-L6-v2 embedder). It also installs `ipykernel` so you can select `.venv` as this notebook's kernel:

```bash
bash setup_env.sh
source .venv/bin/activate && python -m ipykernel install --user --name cisc451 --display-name "CISC451 (.venv)"
```

Then open this notebook with the **CISC451 (.venv)** kernel. The setup script skips steps that are already done, so it's safe to rerun.

## 1. Configuration

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
from IPython.display import Markdown, display

# repo root = the first parent directory that holds run_slurm.sh
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_slurm.sh").exists())

MODE = "slurm"        # "slurm": submit each arm as a GPU job via run_slurm.sh | "local": run here (needs a GPU)
RUN_ID = "repro"      # every arm writes to agent_ext/results/<arm>/<RUN_ID>/
LIMIT = None          # None = full dev (1034); a small number (e.g. 20) for a quick check
N_TRACES = 5          # random traces saved per arm, from successes and from failures each
SEED = 0
BATCH_SIZE = 64       # questions generated together; halved automatically on CUDA out-of-memory
SLURM_GPU = "h100"    # a full H100 (80 GB): batch 64 with long ReAct contexts does not fit a 40 GB MIG slice
ARMS = ["base", "rag", "react", "react_parse", "rag_react", "rag_react_parse"]
SLURM_TIME = {"base": "01:00:00", "rag": "01:00:00"}   # ReAct arms default to the value below
SLURM_TIME_REACT = "03:00:00"

print("repo:", REPO)

## 2. Run the arms

In `slurm` mode, this cell submits all six arms as independent jobs and returns immediately, and the next cell waits for them. In `local` mode it runs them one after another in this kernel's machine and streams the logs.

Rerunning with the same `RUN_ID` overwrites that run. Add `--resume` to `extra` to continue an interrupted run instead.

In [ ]:
def arm_args(arm, extra=()):
    a = ["-m", "agent_ext.run", "--arm", arm, "--run-id", RUN_ID, "--seed", str(SEED),
         "--save-traces", str(N_TRACES), "--batch-size", str(BATCH_SIZE), "--local-only", "--models-path", "a1_lib/models", *extra]
    return a + (["--limit", str(LIMIT)] if LIMIT else [])

jobs = {}
for arm in ARMS:
    if MODE == "slurm":
        env = {**os.environ, "GPU": SLURM_GPU, "TIME": SLURM_TIME.get(arm, SLURM_TIME_REACT)}
        out = subprocess.run(["bash", "run_slurm.sh", *arm_args(arm)], cwd=REPO, env=env,
                             capture_output=True, text=True, check=True).stdout.strip()
        jobs[arm] = out.split()[2]          # "submitted job <id> -> <log>"
        print(arm, "->", out)
    else:
        env = {**os.environ, "PYTHONPATH": str(REPO / "a1_lib"), "HF_HUB_OFFLINE": "1",
               "TRANSFORMERS_OFFLINE": "1", "NLTK_DATA": str(REPO / "nltk_data")}
        print(f"=== {arm}", flush=True)
        subprocess.run([sys.executable, *arm_args(arm)], cwd=REPO, env=env, check=True)

In [ ]:
# slurm mode: wait until every submitted job has left the queue, then report how each ended
if MODE == "slurm" and jobs:
    while True:
        q = subprocess.run(["squeue", "-h", "-j", ",".join(jobs.values()), "-o", "%i %T"],
                           capture_output=True, text=True).stdout.split("\n")
        live = [l for l in q if l.strip()]
        print(time.strftime("%H:%M:%S"), f"{len(live)} job(s) still queued/running", flush=True)
        if not live:
            break
        time.sleep(120)
    for arm, job in jobs.items():
        state = subprocess.run(["sacct", "-n", "-X", "-j", job, "-o", "State"], capture_output=True, text=True).stdout.strip()
        print(f"{arm:16s} job {job}: {state}   log: logs/slurm_run_{job}.log")

## 3. Results

`summary.csv` has one row per finished run. It is appended to, so earlier `RUN_ID`s appear too.

In [ ]:
import pandas as pd
summary = pd.read_csv(REPO / "agent_ext/results/summary.csv")
summary[summary["run"] == RUN_ID]

The comparison script pairs the arms question by question. It reports:

- exact match by difficulty;
- McNemar tests: *gained* counts questions that A gets right and B gets wrong, and *lost* counts the reverse;
- habit diagnostics: whether the query parses in Spider's grader, whether it uses a set operation when the gold does, whether it selects extra columns, and how the ReAct loop behaved.

In [ ]:
env = {**os.environ, "PYTHONPATH": str(REPO / "a1_lib"), "NLTK_DATA": str(REPO / "nltk_data")}
report = subprocess.run([sys.executable, "-m", "agent_ext.compare", "--run-id", RUN_ID], cwd=REPO, env=env,
                        capture_output=True, text=True, check=True).stdout
display(Markdown(report))

**Sanity check.** On the full dev set, the `base` arm should come out close to the original A1 run (`a1_lib/eval/results/Qwen__Qwen3-4B/full_dev`, exact match 0.256), because it uses the same prompt, model and greedy decoding. It may not match exactly: A1 ran at batch size 1, and in a padded bf16 batch a few greedy outputs can change.

In [ ]:
import json
a1 = json.loads((REPO / "a1_lib/eval/results/Qwen__Qwen3-4B/full_dev/metrics.json").read_text())
base = json.loads((REPO / f"agent_ext/results/base/{RUN_ID}/metrics.json").read_text())
for name, m in [("A1 full_dev", a1), (f"base/{RUN_ID}", base)]:
    print(f"{name:20s}", {k: round(v, 3) for k, v in m["exact matching accuracy"]["exact match"].items()})

## 4. Sampled traces

Each run saves `N_TRACES` random successes and failures to `agent_ext/results/<arm>/<RUN_ID>/traces/`, as `success.jsonl`, `failure.jsonl` and a readable `sample.md`. Pick an arm to view its sample.

To resample with a different size or seed without rerunning the model:

```bash
python -m agent_ext.traces agent_ext/results/<arm>/<RUN_ID> --n 10 --seed 1
```

In [ ]:
ARM_TO_VIEW = "rag_react_parse"
display(Markdown((REPO / f"agent_ext/results/{ARM_TO_VIEW}/{RUN_ID}/traces/sample.md").read_text()))